In [0]:
%sql
WITH dx AS (
  SELECT
    person_id,
    -- LEAST in Spark skips NULLs, so this is the earlier of the two starts,
    -- falling back to whichever one exists (matches your extraction anchor)
    LEAST(CAST(inferred_episode_start AS DATE), CAST(recorded_episode_start AS DATE)) AS episode_start,
    CAST(earliest_apo_dx_date AS DATE) AS dx_date
  FROM catalog.schema.step7_filtered_cohort_with_apos
  WHERE earliest_apo_dx_date IS NOT NULL
),
wk AS (
  SELECT
    person_id,
    -- completed weeks since episode start; FLOOR so a negative gap lands in a negative week, not week 0
    FLOOR(DATEDIFF(dx_date, episode_start) / 7) AS dx_week
  FROM dx
  WHERE episode_start IS NOT NULL
)
SELECT
  dx_week,
  COUNT(*) AS n_dx,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct,
  ROUND(100.0 * SUM(COUNT(*)) OVER (ORDER BY dx_week) / SUM(COUNT(*)) OVER (), 2) AS cum_pct
FROM wk
GROUP BY dx_week
ORDER BY dx_week;

In [0]:
%sql
WITH dx AS (
  SELECT
    person_id,
    LEAST(CAST(inferred_episode_start AS DATE), CAST(recorded_episode_start AS DATE)) AS episode_start,
    CAST(earliest_apo_dx_date AS DATE) AS dx_date
  FROM catalog.schema.step7_filtered_cohort_with_apos
),
wk AS (
  SELECT person_id, DATEDIFF(dx_date, episode_start) / 7.0 AS dx_week
  FROM dx
  WHERE dx_date IS NOT NULL AND episode_start IS NOT NULL
)
SELECT
  (SELECT COUNT(*) FROM dx)                                   AS n_people,
  (SELECT COUNT(*) FROM dx WHERE dx_date IS NOT NULL)         AS n_with_dx,
  (SELECT COUNT(*) FROM dx WHERE dx_date IS NOT NULL AND episode_start IS NULL) AS n_dx_no_start,
  COUNT(*)                                                    AS n_analyzed,
  SUM(CASE WHEN dx_week < 0 THEN 1 ELSE 0 END)                AS n_dx_before_start,
  SUM(CASE WHEN dx_week > 42 THEN 1 ELSE 0 END)               AS n_dx_after_42wk,
  ROUND(MIN(dx_week), 1)                                      AS min_wk,
  ROUND(PERCENTILE(dx_week, 0.05), 1)                         AS p05,
  ROUND(PERCENTILE(dx_week, 0.25), 1)                         AS p25,
  ROUND(PERCENTILE(dx_week, 0.50), 1)                         AS median,
  ROUND(PERCENTILE(dx_week, 0.75), 1)                         AS p75,
  ROUND(PERCENTILE(dx_week, 0.95), 1)                         AS p95,
  ROUND(MAX(dx_week), 1)                                      AS max_wk
FROM wk;

In [0]:
%sql
WITH base AS (
  SELECT
    person_id,
    LEAST(CAST(inferred_episode_start AS DATE), CAST(recorded_episode_start AS DATE)) AS episode_start,
    CAST(earliest_apo_dx_date AS DATE) AS dx_date,
    COALESCE(CAST(has_gdm AS INT), 0) = 1 AS gdm,
    -- second outcome = any of the three PE/GHTN flags
    (COALESCE(CAST(has_ghtn AS INT), 0) = 1
     OR COALESCE(CAST(has_pe AS INT), 0) = 1
     OR COALESCE(CAST(has_inferred_pe AS INT), 0) = 1) AS pe_ghtn
  FROM catalog.schema.step7_filtered_cohort_with_apos
  WHERE earliest_apo_dx_date IS NOT NULL
),
grp AS (
  SELECT
    person_id,
    CASE WHEN gdm AND pe_ghtn THEN 'both'
         WHEN gdm             THEN 'gdm_only'
         WHEN pe_ghtn         THEN 'pe_ghtn_only'
         ELSE 'no_flag' END AS outcome_group,
    FLOOR(DATEDIFF(dx_date, episode_start) / 7) AS dx_week
  FROM base
  WHERE episode_start IS NOT NULL
)
SELECT
  outcome_group,
  dx_week,
  COUNT(*) AS n_dx,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY outcome_group), 2) AS pct,
  ROUND(100.0 * SUM(COUNT(*)) OVER (PARTITION BY outcome_group ORDER BY dx_week)
        / SUM(COUNT(*)) OVER (PARTITION BY outcome_group), 2) AS cum_pct
FROM grp
GROUP BY outcome_group, dx_week
ORDER BY outcome_group, dx_week;

In [0]:
%sql
WITH base AS (
  SELECT
    LEAST(CAST(inferred_episode_start AS DATE), CAST(recorded_episode_start AS DATE)) AS episode_start,
    CAST(earliest_apo_dx_date AS DATE) AS dx_date,
    COALESCE(CAST(has_gdm AS INT), 0) = 1 AS gdm,
    (COALESCE(CAST(has_ghtn AS INT), 0) = 1
     OR COALESCE(CAST(has_pe AS INT), 0) = 1
     OR COALESCE(CAST(has_inferred_pe AS INT), 0) = 1) AS pe_ghtn
  FROM catalog.schema.step7_filtered_cohort_with_apos
  WHERE earliest_apo_dx_date IS NOT NULL
),
grp AS (
  SELECT
    CASE WHEN gdm AND pe_ghtn THEN 'both'
         WHEN gdm             THEN 'gdm_only'
         WHEN pe_ghtn         THEN 'pe_ghtn_only'
         ELSE 'no_flag' END AS outcome_group,
    DATEDIFF(dx_date, episode_start) / 7.0 AS dx_week
  FROM base
  WHERE episode_start IS NOT NULL
)
SELECT
  outcome_group,
  COUNT(*)                                       AS n,
  SUM(CASE WHEN dx_week < 1 THEN 1 ELSE 0 END)   AS n_week0,
  SUM(CASE WHEN dx_week > 42 THEN 1 ELSE 0 END)  AS n_after_42wk,
  ROUND(PERCENTILE(dx_week, 0.05), 1)            AS p05,
  ROUND(PERCENTILE(dx_week, 0.25), 1)            AS p25,
  ROUND(PERCENTILE(dx_week, 0.50), 1)            AS median,
  ROUND(PERCENTILE(dx_week, 0.75), 1)            AS p75,
  ROUND(PERCENTILE(dx_week, 0.95), 1)            AS p95
FROM grp
GROUP BY outcome_group
ORDER BY outcome_group;

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CFG = {
    "metrics_table": "catalog.schema.protopregnet_landmark_metrics",
    "metric": "auroc",
    "colors": {"control": "#99bee0", "gdm": "#dbc4a2", "pe_ghtn": "#b1aaec"},
    "labels": {"control": "Control", "gdm": "GDM", "pe_ghtn": "PE/GHTN"},
    "show_ci": True,             # shaded 95% bootstrap CI from ci_lo / ci_hi
    "xtick_step": 1,             # label every Nth week on the x axis
    "figsize": (11, 4.5),
    "save_dir": None,            # e.g. "/path/to/exports"; None = don't save
    # Font sizes centralized here so every text element can be scaled from one place.
    # Tick labels are the most likely to collide (38 weekly ticks on an 11in axis),
    # so raise xtick_step or lower tick_fontsize if they overlap.
    "fonts": {
        "title": 18,
        "axis_label": 16,
        "tick": 12,
        "legend": 14,
    },
}

# ---- load: one row per (model_id, scope, class_name, metric) -> keep AUROC only ----
df = (spark.table(CFG["metrics_table"])
      .filter(f"metric = '{CFG['metric']}'")
      .select("cutoff_week", "scope", "class_name", "estimate", "ci_lo", "ci_hi")
      .toPandas()
      .sort_values("cutoff_week"))

weeks_done = sorted(df["cutoff_week"].unique())
weeks_missing = sorted(set(range(0, 38)) - set(weeks_done))
print(f"{len(weeks_done)} cutoffs found; missing: {weeks_missing or 'none'}")

def plot_series(ax, sub, color, label):
    """One line (+ optional CI band) over cutoff weeks."""
    ax.plot(sub["cutoff_week"], sub["estimate"], marker="o", ms=4, lw=1.8, color=color, label=label)
    if CFG["show_ci"]:
        ax.fill_between(sub["cutoff_week"], sub["ci_lo"], sub["ci_hi"], color=color, alpha=0.15, lw=0)

def style_axes(ax, title):
    f = CFG["fonts"]
    ax.set_xlabel("Landmark (weeks since episode start)", fontsize=f["axis_label"])
    ax.set_ylabel("AUROC", fontsize=f["axis_label"])
    ax.set_title(title, fontsize=f["title"])
    ax.set_xticks(range(0, 38, CFG["xtick_step"]))
    ax.tick_params(axis="both", labelsize=f["tick"])
    ax.grid(alpha=0.3)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

# ---- plot 1: overall macro AUROC (mean over control, pe_ghtn, gdm) ----
fig1, ax1 = plt.subplots(figsize=CFG["figsize"])
plot_series(ax1, df[df["scope"] == "macro"], "#7cc37c", "Macro AUROC")
style_axes(ax1, "Overall macro AUROC by landmark (held-out test)")
ax1.legend(frameon=False, fontsize=CFG["fonts"]["legend"])
fig1.tight_layout()

# ---- plot 2: outcome-specific AUROC, one color per class ----
fig2, ax2 = plt.subplots(figsize=CFG["figsize"])
for cls in ["control", "gdm", "pe_ghtn"]:
    plot_series(ax2, df[(df["scope"] == "class") & (df["class_name"] == cls)],
                CFG["colors"][cls], CFG["labels"][cls])
style_axes(ax2, "Outcome-specific AUROC by landmark (held-out test)")
ax2.legend(frameon=False, fontsize=CFG["fonts"]["legend"])
fig2.tight_layout()

if CFG["save_dir"]:
    fig1.savefig(f"{CFG['save_dir']}/landmark_auroc_macro.png", dpi=800, bbox_inches="tight")
    fig2.savefig(f"{CFG['save_dir']}/landmark_auroc_by_outcome.png", dpi=800, bbox_inches="tight")
plt.show()

In [0]:
%sql
SELECT *
FROM catalog.schema.protopregnet_landmark_metrics
WHERE metric = 'auroc' and cutoff_week = 37

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CFG = {
    "metrics_table": "catalog.schema.protopregnet_landmark_metrics",
    "metric": "auroc",
    "colors": {"control": "#99bee0", "gdm": "#dbc4a2", "pe_ghtn": "#b1aaec"},
    "labels": {"control": "Control", "gdm": "GDM", "pe_ghtn": "PE/GHTN"},
    "macro_color": "#7cc37c",
    "xtick_step": 1,             # label every Nth week on the x axis
    "figsize": (11, 5.5),        # slightly taller than before: four lines share one axis
    "save_dir": None,            # e.g. "/path/to/exports"; None = don't save
    # Visual hierarchy: macro is the headline series, so it gets a thicker, opaque,
    # solid line drawn on top (high zorder), plus its 95% bootstrap CI band.
    # Outcome-specific series are dashed, thinner, partially transparent, and have
    # no CI band (show_ci=False) so they read as supporting context without clutter.
    "macro_style": {"lw": 3.0, "ms": 6, "alpha": 1.0, "ls": "-", "zorder": 5,
                    "show_ci": True, "ci_alpha": 0.20},
    "class_style": {"lw": 1.6, "ms": 3.5, "alpha": 0.7, "ls": "--", "zorder": 3,
                    "show_ci": False, "ci_alpha": 0.0},
    "fonts": {
        "title": 18,
        "axis_label": 16,
        "tick": 12,
        "legend": 14,
    },
}

# ---- load: one row per (model_id, scope, class_name, metric) -> keep AUROC only ----
df = (spark.table(CFG["metrics_table"])
      .filter(f"metric = '{CFG['metric']}'")
      .select("cutoff_week", "scope", "class_name", "estimate", "ci_lo", "ci_hi")
      .toPandas()
      .sort_values("cutoff_week"))

weeks_done = sorted(df["cutoff_week"].unique())
weeks_missing = sorted(set(range(0, 38)) - set(weeks_done))
print(f"{len(weeks_done)} cutoffs found; missing: {weeks_missing or 'none'}")

def plot_series(ax, sub, color, label, style):
    """One line (+ optional CI band) over cutoff weeks, styled per CFG style dict."""
    ax.plot(sub["cutoff_week"], sub["estimate"], marker="o", ms=style["ms"], lw=style["lw"],
            ls=style["ls"], alpha=style["alpha"], color=color, label=label, zorder=style["zorder"])
    if style["show_ci"]:
        # CI band sits below the line (zorder - 1) so the line stays readable on top.
        ax.fill_between(sub["cutoff_week"], sub["ci_lo"], sub["ci_hi"], color=color,
                        alpha=style["ci_alpha"], lw=0, zorder=style["zorder"] - 1)

def style_axes(ax, title):
    f = CFG["fonts"]
    ax.set_xlabel("Landmark (weeks since episode start)", fontsize=f["axis_label"])
    ax.set_ylabel("AUROC", fontsize=f["axis_label"])
    ax.set_title(title, fontsize=f["title"])
    ax.set_xticks(range(0, 38, CFG["xtick_step"]))
    ax.tick_params(axis="both", labelsize=f["tick"])
    ax.grid(alpha=0.3)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

# ---- single combined plot: outcome-specific (dashed, no CI) under macro (solid, bold, with CI) ----
fig, ax = plt.subplots(figsize=CFG["figsize"])

# Draw class series first so macro is layered on top (zorder also enforces this).
for cls in ["control", "gdm", "pe_ghtn"]:
    plot_series(ax, df[(df["scope"] == "class") & (df["class_name"] == cls)],
                CFG["colors"][cls], CFG["labels"][cls], CFG["class_style"])

plot_series(ax, df[df["scope"] == "macro"], CFG["macro_color"], "Macro AUROC", CFG["macro_style"])

style_axes(ax, "Macro and outcome-specific AUROC by landmark (held-out test)")

# Reorder legend so Macro comes first, then the classes.
handles, labels = ax.get_legend_handles_labels()
order = [labels.index("Macro AUROC")] + [i for i, l in enumerate(labels) if l != "Macro AUROC"]
ax.legend([handles[i] for i in order], [labels[i] for i in order],
          frameon=False, fontsize=CFG["fonts"]["legend"])
fig.tight_layout()

if CFG["save_dir"]:
    fig.savefig(f"{CFG['save_dir']}/landmark_auroc_combined.png", dpi=800, bbox_inches="tight")
plt.show()

In [0]:
import numpy as np, pandas as pd
import pyspark.sql.functions as sf
from sklearn.metrics import roc_auc_score

CFG = {
    "meds_path": "/path/to/exports/meds_output_spark/data",
    "cohort_table": "catalog.schema.protopregnet_landmark_cohort",
    "v5_global_table": "catalog.schema.patient_clmbr_embeddings_new_v5",
    "split": "test",
}

coh = (spark.table(CFG["cohort_table"]).filter(sf.col("split") == CFG["split"])
       .select("pid_int", "y_control", "y_pe_ghtn", "y_gdm", "episode_start").toPandas())
last_ev = (spark.read.parquet(CFG["meds_path"]).select(sf.col("subject_id").alias("pid_int"), "time")
           .groupBy("pid_int").agg(sf.max("time").alias("last_event")).toPandas())
seq = (spark.table(CFG["v5_global_table"])
       .select(sf.col("patient_id").alias("pid_int"), "sequence_length").toPandas())

d = coh.merge(last_ev, on="pid_int").merge(seq, on="pid_int")
d["last_event_week"] = (pd.to_datetime(d["last_event"]) - pd.to_datetime(d["episode_start"])).dt.days / 7

rows = [(f, c, roc_auc_score(d[c], d[f])) for f in ["last_event_week", "sequence_length"]
        for c in ["y_control", "y_pe_ghtn", "y_gdm"]]
print(pd.DataFrame(rows, columns=["single feature", "class", "AUROC"]).round(3).to_string(index=False))
print(d.groupby("y_control")["last_event_week"].describe()[["25%", "50%", "75%"]].round(1))

In [0]:
%sql
INSERT INTO catalog.schema.protopregnet_landmark_eligibility
SELECT pid_int, 99 FROM catalog.schema.protopregnet_landmark_cohort

In [0]:
import numpy as np, pandas as pd
import pyspark.sql.functions as sf
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

CFG = {
    "meds_path": "/path/to/exports/meds_output_spark/data",
    "cohort_table": "catalog.schema.protopregnet_landmark_cohort",
    "v5_global_table": "catalog.schema.patient_clmbr_embeddings_new_v5",
    "alphas": [1.0, 10.0, 100.0, 1000.0],
}

coh = spark.table(CFG["cohort_table"]).select("pid_int", "split", "y_control", "episode_start").toPandas()
last_ev = (spark.read.parquet(CFG["meds_path"]).select(sf.col("subject_id").alias("pid_int"), "time")
           .groupBy("pid_int").agg(sf.max("time").alias("last_event")).toPandas())
emb = (spark.table(CFG["v5_global_table"]).withColumnRenamed("patient_id", "pid_int")
       .drop("sequence_length").toPandas())

d = coh.merge(last_ev, on="pid_int").merge(emb, on="pid_int")
d["last_event_week"] = (pd.to_datetime(d["last_event"]) - pd.to_datetime(d["episode_start"])).dt.days / 7
emb_cols = [c for c in d.columns if c.startswith("emb_")]

tr, te = d[d["split"] == "train"], d[d["split"] == "test"]
ridge = RidgeCV(alphas=CFG["alphas"]).fit(tr[emb_cols].to_numpy(), tr["last_event_week"].to_numpy())
pred = ridge.predict(te[emb_cols].to_numpy())
print(f"R2 overall: {r2_score(te['last_event_week'], pred):.3f}")
for name, mask in [("controls", te["y_control"] == 1), ("cases", te["y_control"] == 0)]:
    print(f"R2 within {name}: {r2_score(te.loc[mask, 'last_event_week'], pred[mask.to_numpy()]):.3f}")

In [0]:
!pip install torch

In [0]:
import pickle
import numpy as np
import pandas as pd
import pyspark.sql.functions as sf
from sklearn.metrics import roc_auc_score

CFG = {
    "cohort_table": "catalog.schema.protopregnet_landmark_cohort",
    "meds_path": "/path/to/exports/meds_output_spark/data",
    "orig_export": "/path/to/exports/full_analysis_export.pkl",
    "landmark_export": "/path/to/exports/full_analysis_export_landmark_wk{kk:02d}.pkl",
    "cutoffs": [10, 20, 30, 37],                                # landmarks used for the lead-time test (Test A)
    "record_end_bins": [-np.inf, 28, 32, 36, 40, np.inf],       # weeks since episode start (Test B)
    "lead_bins": [0, 1, 2, 4, 8, np.inf],                       # weeks from the cutoff to the first APO dx (Test A)
    "min_pos": 15, "min_neg": 50,                               # skip strata too small for a stable AUROC
}
CLASSES = ["control", "pe_ghtn", "gdm"]


def auc(y, p):
    return roc_auc_score(y, p)


# ---- original model's test predictions (full-record embeddings) ----
with open(CFG["orig_export"], "rb") as f:
    o = pickle.load(f)
orig = pd.DataFrame(np.asarray(o["p_test"]), columns=[f"p_{c}" for c in CLASSES])
orig["pid_int"] = np.asarray(o["pids_test"]).astype("int64")
del o

# ---- labels, episode start, first APO date ----
coh = (spark.table(CFG["cohort_table"])
       .select("pid_int", "y_control", "y_pe_ghtn", "y_gdm", "episode_start", "earliest_apo_dx_date").toPandas())

# ---- last MEDS event per test patient (when each record ends); filtered to the test patients first ----
ids = spark.createDataFrame(orig[["pid_int"]])
last_ev = (spark.read.parquet(CFG["meds_path"]).select(sf.col("subject_id").alias("pid_int"), "time")
           .join(sf.broadcast(ids), "pid_int").groupBy("pid_int")
           .agg(sf.max("time").alias("last_event")).toPandas())

d = orig.merge(coh, on="pid_int").merge(last_ev, on="pid_int").reset_index(drop=True)
d["end_wk"] = (pd.to_datetime(d["last_event"]) - pd.to_datetime(d["episode_start"])).dt.days / 7
print(f"{len(d)} test patients with original predictions, labels and record end")

In [0]:
# Compare the original model's AUROC overall vs WITHIN strata of record-end week. Within a stratum, cases and
# controls have records ending at about the same gestational age, so record length can't separate them.
d["end_bin"] = pd.cut(d["end_wk"], CFG["record_end_bins"], right=False)

rows, summary = [], []
for c in CLASSES:
    used_idx, weights, aucs = [], [], []
    for b, g in d.groupby("end_bin", observed=True):
        yb = g[f"y_{c}"].to_numpy(); n_pos = int(yb.sum()); n_neg = len(yb) - n_pos
        ok = n_pos >= CFG["min_pos"] and n_neg >= CFG["min_neg"]
        a = auc(yb, g[f"p_{c}"]) if ok else np.nan
        rows.append({"class": c, "record_end_week": str(b), "n": len(g), "n_pos": n_pos, "auroc": a})
        if ok:
            used_idx.append(g.index.to_numpy()); weights.append(n_pos * n_neg); aucs.append(a)
    overall = auc(d[f"y_{c}"], d[f"p_{c}"])
    if used_idx:
        idx = np.concatenate(used_idx)
        overall_used = auc(d.loc[idx, f"y_{c}"], d.loc[idx, f"p_{c}"])
        pooled = float(np.average(aucs, weights=weights))     # pair-weighted mean of within-stratum AUROCs
    else:
        overall_used = pooled = np.nan
    summary.append({"class": c, "overall": overall, "overall_same_patients": overall_used,
                    "within_record_end_strata": pooled, "n_strata": len(aucs)})

print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\n", pd.DataFrame(summary).round(3).to_string(index=False))

In [0]:
# For each landmark: AUROC of the LANDMARK model vs the ORIGINAL model on the SAME patients, with positives
# split by how long after the cutoff they are diagnosed. Positives = single-outcome cases of the class (so the
# first-APO date is that class's date); negatives = everyone eligible with y_class = 0.
rows = []
for k in CFG["cutoffs"]:
    # with open(CFG["landmark_export"].format(kk=k), "rb") as f:
    #     L = pickle.load(f)
    L = pd.read_pickle(CFG["landmark_export"].format(kk=k))
    lm = pd.DataFrame(np.asarray(L["p_test"]), columns=[f"lm_{c}" for c in CLASSES])
    lm["pid_int"] = np.asarray(L["pids_test"]).astype("int64")
    del L
    m = lm.merge(d, on="pid_int")
    cutoff = pd.to_datetime(m["episode_start"]) + pd.to_timedelta(7 * k, unit="D")
    m["lead_wk"] = (pd.to_datetime(m["earliest_apo_dx_date"]) - cutoff).dt.days / 7

    for c, other in [("pe_ghtn", "gdm"), ("gdm", "pe_ghtn")]:
        neg = m[m[f"y_{c}"] == 0]
        pos_all = m[(m[f"y_{c}"] == 1) & (m[f"y_{other}"] == 0)]
        for lo, hi in zip(CFG["lead_bins"][:-1], CFG["lead_bins"][1:]):
            pos = pos_all[(pos_all["lead_wk"] >= lo) & (pos_all["lead_wk"] < hi)]
            if len(pos) < CFG["min_pos"] or len(neg) < CFG["min_neg"]:
                a_lm = a_or = np.nan
            else:
                sub = pd.concat([pos, neg]); y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
                a_lm, a_or = auc(y, sub[f"lm_{c}"]), auc(y, sub[f"p_{c}"])
            rows.append({"cutoff": k, "class": c, "lead_wk": f"[{lo},{hi})", "n_pos": len(pos),
                         "auroc_landmark": a_lm, "auroc_original": a_or})

res = pd.DataFrame(rows)
print(res.round(3).to_string(index=False))

In [0]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(CFG["cutoffs"])))
for ax, c in zip(axes, ["pe_ghtn", "gdm"]):
    for col, k in zip(colors, CFG["cutoffs"]):
        s = res[(res["class"] == c) & (res["cutoff"] == k)]
        ax.plot(s["lead_wk"], s["auroc_landmark"], marker="o", color=col, label=f"landmark wk{k}")
        ax.plot(s["lead_wk"], s["auroc_original"], ls="--", color=col, alpha=0.6)
    ax.set_title(f"{c}: AUROC by weeks until diagnosis (dashed = original model, same patients)")
    ax.set_xlabel("Weeks from landmark to diagnosis"); ax.grid(alpha=0.3)
    ax.title.set_fontsize(9)
axes[0].set_ylabel("AUROC"); axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

In [0]:
import pyspark.sql.functions as sf
meds = spark.read.parquet("/path/to/exports/meds_output_spark/data")
print(meds.columns)
if "numeric_value" in meds.columns:
    print("share of rows with a numeric value:",
          meds.select(sf.avg(sf.col("numeric_value").isNotNull().cast("int"))).first()[0])

In [0]:
# Early- vs late-onset PE/GHTN: AUROC of landmark vs original model by gestational age AT DIAGNOSIS.
CFG_EO = {
    "landmarks": [10, 20, 30],
    "onset_bins": [(0, 34), (34, 37), (37, np.inf)],      # gestational weeks at diagnosis; <34 = conventional "early-onset"
}
rows = []
for k in CFG_EO["landmarks"]:
    L = pd.read_pickle(CFG["landmark_export"].format(kk=k))
    lm = pd.DataFrame(np.asarray(L["p_test"]), columns=[f"lm_{c}" for c in CLASSES])
    lm["pid_int"] = np.asarray(L["pids_test"]).astype("int64")
    del L
    m = lm.merge(d, on="pid_int")
    m["dx_wk"] = (pd.to_datetime(m["earliest_apo_dx_date"]) - pd.to_datetime(m["episode_start"])).dt.days / 7
    neg = m[m["y_pe_ghtn"] == 0]
    pos_all = m[(m["y_pe_ghtn"] == 1) & (m["y_gdm"] == 0)]
    for lo, hi in CFG_EO["onset_bins"]:
        pos = pos_all[(pos_all["dx_wk"] >= lo) & (pos_all["dx_wk"] < hi)]
        if len(pos) < CFG["min_pos"]:
            a_lm = a_or = np.nan
        else:
            sub = pd.concat([pos, neg]); y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
            a_lm, a_or = auc(y, sub["lm_pe_ghtn"]), auc(y, sub["p_pe_ghtn"])
        rows.append({"landmark": k, "dx_week": f"[{lo},{hi})", "n_pos": len(pos),
                     "auroc_landmark": a_lm, "auroc_original": a_or})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [0]:
# Fixed-horizon AUROC computed from the EXISTING open-horizon landmark predictions (no retraining).
# Label for class c at landmark k: positive = c-only case diagnosed within H weeks of the landmark; negative = every other
# eligible patient (controls, other-outcome cases, AND c cases diagnosed later than H). Patients with both outcomes are
# dropped because the first-APO date can't be attributed to one outcome.
CFG_FH = {"landmarks": [10, 20, 30, 37], "horizons_wk": [2, 4, 8], "min_pos": 15}
rows = []
for k in CFG_FH["landmarks"]:
    L = pd.read_pickle(CFG["landmark_export"].format(kk=k))
    lm = pd.DataFrame(np.asarray(L["p_test"]), columns=[f"lm_{c}" for c in CLASSES])
    lm["pid_int"] = np.asarray(L["pids_test"]).astype("int64")
    del L
    m = lm.merge(d, on="pid_int")
    m = m[~((m["y_pe_ghtn"] == 1) & (m["y_gdm"] == 1))]
    m["lead_wk"] = (pd.to_datetime(m["earliest_apo_dx_date"])
                    - (pd.to_datetime(m["episode_start"]) + pd.to_timedelta(7 * k, unit="D"))).dt.days / 7
    for c, other in [("pe_ghtn", "gdm"), ("gdm", "pe_ghtn")]:
        c_only = ((m[f"y_{c}"] == 1) & (m[f"y_{other}"] == 0)).to_numpy()
        auroc_open = auc(c_only, m[f"lm_{c}"]) if c_only.sum() >= CFG["min_pos"] else np.nan
        for H in CFG_FH["horizons_wk"]:
            y = c_only & (m["lead_wk"] <= H).to_numpy()
            ok = y.sum() >= CFG["min_pos"]
            rows.append({"landmark": k, "class": c, "horizon_wk": H, "n_pos": int(y.sum()),
                         "prevalence": round(float(y.mean()), 4),
                         "auroc_fixed": auc(y, m[f"lm_{c}"]) if ok else np.nan, "auroc_open": auroc_open})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CFG = {
    # "metrics_table": "catalog.schema.protopregnet_rf_landmark_metrics",
    "metrics_table": "catalog.schema.protopregnet_rf_v2_landmark_metrics",
    "model_id_prefix": "rf_v2_landmark_wk",       # change if you used another model_id_template in the RF notebook
    "metric": "auroc",
    "landmarks": list(range(0, 38)),
    "colors": {"control": "#7f7f7f", "gdm": "#1f77b4", "pe_ghtn": "#d62728"},
    "labels": {"control": "Control", "gdm": "GDM", "pe_ghtn": "PE/GHTN"},
    "show_ci": True,             # shaded 95% bootstrap CI for the RF curves (ci_lo / ci_hi)
    # optional: draw ProtoPregNet's landmark curves (dashed) on the same axes for comparison
    "overlay_ppn": True,
    "show_ci_ppn": True,        # shaded 95% CI for the ProtoPregNet overlay (independent of show_ci)
    "ci_alpha": 0.15, "ppn_ci_alpha": 0.10,   # band opacity; the overlay's is lighter so both stay readable when stacked
    "ppn_metrics_table": "catalog.schema.protopregnet_landmark_metrics",
    "ppn_model_id_prefix": "landmark_wk",
    "xtick_step": 1,             # label every Nth week on the x axis
    "figsize": (11, 4.5),
    "save_dir": None,            # e.g. "/path/to/exports"; None = don't save
}


def load_metrics(table, prefix):
    """One row per (cutoff_week, scope, class_name) for the chosen metric. Filtering happens in pandas (the table is small), so
    replication/washout/other model_ids in the same table can't leak in, and duplicates are caught instead of drawn twice."""
    df = (spark.table(table)
          .select("model_id", "cutoff_week", "scope", "class_name", "metric", "estimate", "ci_lo", "ci_hi", "n_test")
          .toPandas())
    df = df[(df["metric"] == CFG["metric"]) & df["model_id"].astype(str).str.startswith(prefix)].sort_values("cutoff_week")
    if df.empty:
        raise ValueError(f"no '{CFG['metric']}' rows with model_id starting '{prefix}' in {table}")
    dup = df.groupby(["cutoff_week", "scope", "class_name"]).size()
    assert (dup == 1).all(), f"duplicate rows (more than one model_id per landmark?): {dup[dup > 1].index.tolist()[:5]}"
    return df


df = load_metrics(CFG["metrics_table"], CFG["model_id_prefix"])
ppn = load_metrics(CFG["ppn_metrics_table"], CFG["ppn_model_id_prefix"]) if CFG["overlay_ppn"] else None

weeks_done = sorted(df["cutoff_week"].unique())
weeks_missing = sorted(set(CFG["landmarks"]) - set(weeks_done))
print(f"{len(weeks_done)} RF cutoffs found; missing: {weeks_missing or 'none'}")


def plot_series(ax, sub, color, label, ci, alpha, ls="-", marker="o"):
    """One line (+ optional CI band) over cutoff weeks."""
    ax.plot(sub["cutoff_week"], sub["estimate"], marker=marker, ms=4, lw=1.8, ls=ls, color=color, label=label)
    if ci:
        ax.fill_between(sub["cutoff_week"], sub["ci_lo"], sub["ci_hi"], color=color, alpha=alpha, lw=0)


def style_axes(ax, title):
    ax.set_xlabel("Landmark (weeks since episode start)")
    ax.set_ylabel("AUROC")
    ax.set_title(title)
    ax.set_xticks(range(min(CFG["landmarks"]), max(CFG["landmarks"]) + 1, CFG["xtick_step"]))
    ax.grid(alpha=0.3)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


# ---- plot 1: overall macro AUROC (mean over control, pe_ghtn, gdm) ----
fig1, ax1 = plt.subplots(figsize=CFG["figsize"])
plot_series(ax1, df[df["scope"] == "macro"], "#2ca02c", "RF macro AUROC" if ppn is not None else "Macro AUROC",
            ci=CFG["show_ci"], alpha=CFG["ci_alpha"], ls="--", marker="s")
if ppn is not None:
    plot_series(ax1, ppn[ppn["scope"] == "macro"], "#2ca02c", "ProtoPregNet macro AUROC",
                ci=CFG["show_ci_ppn"], alpha=CFG["ppn_ci_alpha"])
style_axes(ax1, "ProtoPregNet vs RF: overall macro AUROC by landmark (held-out test)")
ax1.legend(frameon=False)
fig1.tight_layout()

# ---- plot 2: outcome-specific AUROC, one color per class ----
fig2, ax2 = plt.subplots(figsize=CFG["figsize"])
for cls in ["control", "gdm", "pe_ghtn"]:
    plot_series(ax2, df[(df["scope"] == "class") & (df["class_name"] == cls)], CFG["colors"][cls],
                CFG["labels"][cls] + (" (RF)" if ppn is not None else ""), ci=CFG["show_ci"], alpha=CFG["ci_alpha"], ls="--", marker="s")
    if ppn is not None:
        plot_series(ax2, ppn[(ppn["scope"] == "class") & (ppn["class_name"] == cls)], CFG["colors"][cls],
                    CFG["labels"][cls] + " (ProtoPregNet)", ci=CFG["show_ci_ppn"], alpha=CFG["ppn_ci_alpha"])
style_axes(ax2, "ProtoPregNet vs RF: outcome-specific AUROC by landmark (held-out test)")
ax2.legend(frameon=False, ncol=2 if ppn is not None else 1)
fig2.tight_layout()

if CFG["save_dir"]:
    fig1.savefig(f"{CFG['save_dir']}/rf_landmark_auroc_macro.png", dpi=200)
    fig2.savefig(f"{CFG['save_dir']}/rf_landmark_auroc_by_outcome.png", dpi=200)
plt.show()

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CFG = {
    "metric": "auroc",
    "landmarks": list(range(0, 38)),
    "colors": {"control": "#7f7f7f", "gdm": "#1f77b4", "pe_ghtn": "#d62728"},
    "labels": {"control": "Control", "gdm": "GDM", "pe_ghtn": "PE/GHTN"},
    "macro_color": "#2ca02c",

    # ---- RF v2 (always drawn) ----
    "metrics_table": "catalog.schema.protopregnet_rf_v2_landmark_metrics",
    "model_id_prefix": "rf_v2_landmark_wk",       # change if you used another model_id_template in the RF notebook
    "show_ci": True,             # shaded 95% bootstrap CI for the RF curves (ci_lo / ci_hi)
    "ci_alpha": 0.15,

    # ---- XGBoost v2 (optional overlay) ----
    "overlay_xgb": True,
    "xgb_metrics_table": "catalog.schema.protopregnet_xgb_v2_landmark_metrics",
    "xgb_model_id_prefix": "xgb_v2_landmark_wk",
    "show_ci_xgb": True,         # shaded 95% CI for the XGBoost curves (independent of the other CI toggles)
    "xgb_ci_alpha": 0.08,        # lightest of the three, so the bands stay readable when stacked

    # ---- ProtoPregNet (optional overlay) ----
    "overlay_ppn": True,
    "ppn_metrics_table": "catalog.schema.protopregnet_landmark_metrics",
    "ppn_model_id_prefix": "landmark_wk",
    "show_ci_ppn": True,         # shaded 95% CI for the ProtoPregNet overlay
    "ppn_ci_alpha": 0.10,

    # line style and marker per model (macro and per-outcome plots)
    "styles": {"ProtoPregNet": ("-", "o"), "RF": ("--", "s"), "XGBoost": (":", "^")},
    "xtick_step": 1,             # label every Nth week on the x axis
    "figsize": (11, 4.5),
    "save_dir": None,            # e.g. "/path/to/exports"; None = don't save
}


def load_metrics(table, prefix):
    """One row per (cutoff_week, scope, class_name) for the chosen metric. Filtering happens in pandas (the table is small), so
    replication/washout/other model_ids in the same table can't leak in, and duplicates are caught instead of drawn twice."""
    df = (spark.table(table)
          .select("model_id", "cutoff_week", "scope", "class_name", "metric", "estimate", "ci_lo", "ci_hi", "n_test")
          .toPandas())
    df = df[(df["metric"] == CFG["metric"]) & df["model_id"].astype(str).str.startswith(prefix)].sort_values("cutoff_week")
    if df.empty:
        raise ValueError(f"no '{CFG['metric']}' rows with model_id starting '{prefix}' in {table}")
    dup = df.groupby(["cutoff_week", "scope", "class_name"]).size()
    assert (dup == 1).all(), f"duplicate rows (more than one model_id per landmark?): {dup[dup > 1].index.tolist()[:5]}"
    return df


# one entry per model, drawn in this order (later models on top)
series = [
    {"name": "ProtoPregNet", "on": CFG["overlay_ppn"], "table": CFG["ppn_metrics_table"], "prefix": CFG["ppn_model_id_prefix"], "ci": CFG["show_ci_ppn"], "alpha": CFG["ppn_ci_alpha"]},
    {"name": "RF", "on": True, "table": CFG["metrics_table"], "prefix": CFG["model_id_prefix"], "ci": CFG["show_ci"], "alpha": CFG["ci_alpha"]},
    {"name": "XGBoost", "on": CFG["overlay_xgb"], "table": CFG["xgb_metrics_table"], "prefix": CFG["xgb_model_id_prefix"], "ci": CFG["show_ci_xgb"], "alpha": CFG["xgb_ci_alpha"]},
]
series = [s for s in series if s["on"]]
for s in series:
    s["df"] = load_metrics(s["table"], s["prefix"])
    s["ls"], s["marker"] = CFG["styles"][s["name"]]
    missing = sorted(set(CFG["landmarks"]) - set(s["df"]["cutoff_week"].unique()))
    print(f"{s['name']}: {s['df']['cutoff_week'].nunique()} cutoffs found; missing: {missing or 'none'}")
names = " vs ".join(s["name"] for s in series)


def plot_series(ax, sub, color, label, ci, alpha, ls="-", marker="o"):
    """One line (+ optional CI band) over cutoff weeks."""
    ax.plot(sub["cutoff_week"], sub["estimate"], marker=marker, ms=4, lw=1.8, ls=ls, color=color, label=label)
    if ci:
        ax.fill_between(sub["cutoff_week"], sub["ci_lo"], sub["ci_hi"], color=color, alpha=alpha, lw=0)


def style_axes(ax, title):
    ax.set_xlabel("Landmark (weeks since episode start)")
    ax.set_ylabel("AUROC")
    ax.set_title(title)
    ax.set_xticks(range(min(CFG["landmarks"]), max(CFG["landmarks"]) + 1, CFG["xtick_step"]))
    ax.grid(alpha=0.3)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


# ---- plot 1: overall macro AUROC (mean over control, pe_ghtn, gdm) ----
fig1, ax1 = plt.subplots(figsize=CFG["figsize"])
for s in series:
    plot_series(ax1, s["df"][s["df"]["scope"] == "macro"], CFG["macro_color"], f"{s['name']} macro AUROC", ci=s["ci"], alpha=s["alpha"], ls=s["ls"], marker=s["marker"])
style_axes(ax1, f"{names}: overall macro AUROC by landmark (held-out test)")
ax1.legend(frameon=False)
fig1.tight_layout()

# ---- plot 2: outcome-specific AUROC, one color per class, one line style per model ----
fig2, ax2 = plt.subplots(figsize=CFG["figsize"])
for cls in ["control", "gdm", "pe_ghtn"]:
    for s in series:
        plot_series(ax2, s["df"][(s["df"]["scope"] == "class") & (s["df"]["class_name"] == cls)], CFG["colors"][cls],
                    f"{CFG['labels'][cls]} ({s['name']})", ci=s["ci"], alpha=s["alpha"], ls=s["ls"], marker=s["marker"])
style_axes(ax2, f"{names}: outcome-specific AUROC by landmark (held-out test)")
ax2.legend(frameon=False, ncol=len(series))
fig2.tight_layout()

if CFG["save_dir"]:
    fig1.savefig(f"{CFG['save_dir']}/landmark_auroc_macro_comparison.png", dpi=200)
    fig2.savefig(f"{CFG['save_dir']}/landmark_auroc_by_outcome_comparison.png", dpi=200)
plt.show()